# Statistical Thinking for AI Systems — Capstone Practice Notebook

This notebook is a hands-on companion to the Markdown file on
**Statistical Thinking for AI Systems**.
It simulates an end-to-end production AI workflow and integrates concepts from all 20 lectures.

Workflow stages:

1. Data quality and uncertainty check
2. Predictive modeling and probability calibration
3. Online A/B validation
4. Feature drift monitoring
5. Reward and KPI monitoring
6. Failure diagnosis dashboard
7. Trust report summary
8. Mini exercises

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, brier_score_loss
from scipy import stats
np.random.seed(42)

## 1. Data Quality and Uncertainty Check

Before training, assess data quality:
- Missing values: how many and where?
- Global missing rate above a threshold signals a data pipeline issue.

Missing data is a source of **epistemic uncertainty** — it reduces what we can know.

In [ ]:
X_raw, y = make_classification(n_samples=600, n_features=8, n_informative=5, random_state=42)
X_raw = pd.DataFrame(X_raw)

# Simulate 3% random missing values
mask = np.random.rand(*X_raw.shape) < 0.03
X_missing = X_raw.mask(mask)

missing_rate    = X_missing.isna().mean().mean()
missing_by_col  = X_missing.isna().sum()

print(f'Global missing rate: {missing_rate:.4f}')
pd.DataFrame({'Column': X_missing.columns, 'Missing count': missing_by_col.values})

## 2. Predictive Modeling and Probability Calibration

After imputing missing values, we train a classifier and evaluate:
- **Accuracy:** Fraction of correct predictions
- **Brier score:** Mean squared error of probabilities (lower is better)
- **Calibration curve:** Are predicted probabilities trustworthy?

In [ ]:
X_filled = X_missing.fillna(X_missing.mean())
X_train, X_test, y_train, y_test = train_test_split(X_filled, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred  = model.predict(X_test)
prob  = model.predict_proba(X_test)[:, 1]

brier = brier_score_loss(y_test, prob)
acc   = accuracy_score(y_test, pred)

pd.DataFrame({'Metric': ['Accuracy', 'Brier score'], 'Value': [acc, brier]})

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, prob, n_bins=10)
plt.figure(figsize=(6, 5))
plt.plot(mean_pred, frac_pos, marker='o', label='Model')
plt.plot([0, 1], [0, 1], '--', label='Perfect calibration')
plt.title('Calibration Curve')
plt.xlabel('Predicted probability')
plt.ylabel('Observed positive fraction')
plt.legend()
plt.show()

## 3. Online A/B Validation

Before deploying a new model, run a controlled A/B test.
Test whether the new variant B has a significantly different conversion rate than control A.

In [ ]:
A = np.random.binomial(1, 0.12, 500)
B = np.random.binomial(1, 0.16, 520)

lift   = B.mean() - A.mean()
t_stat, p_val = stats.ttest_ind(B, A, equal_var=False)

pd.DataFrame({
    'Metric': ['A rate', 'B rate', 'Lift (B - A)', 'p-value', 'Significant?'],
    'Value':  [round(A.mean(), 4), round(B.mean(), 4), round(lift, 4),
               round(p_val, 4), 'Yes' if p_val < 0.05 else 'No']
})

## 4. Feature Drift Monitoring

In production, input feature distributions can shift over time.
We use the **Kolmogorov-Smirnov test** to detect distributional drift:

$$D_{KS} = \sup_x |F_n(x) - G_m(x)|$$

A small p-value signals a significant shift that may degrade model performance.

In [ ]:
train_feature = X_train.iloc[:, 0]
# Simulate a mean shift of 0.5 in live data
live_feature  = train_feature + np.random.normal(0.5, 0.3, len(train_feature))

ks_stat, ks_p = stats.ks_2samp(train_feature, live_feature)

pd.DataFrame({
    'Metric':  ['KS statistic', 'p-value', 'Drift detected?'],
    'Value':   [round(ks_stat, 4), round(ks_p, 4), 'Yes' if ks_p < 0.05 else 'No']
})

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(train_feature, bins=25, alpha=0.6, label='Train distribution')
plt.hist(live_feature,  bins=25, alpha=0.6, label='Live distribution')
plt.title('Feature Drift Check')
plt.xlabel('Feature value')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## 5. Reward and KPI Monitoring

Track system reward (or business KPI) over time and compute a running confidence interval.
A widening CI or sudden drop can trigger a model review.

In [ ]:
daily_reward = np.random.normal(75, 8, 60)   # 60 days of daily KPI
reward_mean  = daily_reward.mean()
reward_se    = daily_reward.std(ddof=1) / np.sqrt(len(daily_reward))
reward_ci    = 1.96 * reward_se

plt.figure(figsize=(9, 4))
plt.plot(daily_reward, alpha=0.7)
plt.axhline(reward_mean, linestyle='--', label=f'Mean = {reward_mean:.1f}')
plt.fill_between(range(len(daily_reward)),
                 reward_mean - reward_ci, reward_mean + reward_ci,
                 alpha=0.2, label='95% CI for mean')
plt.title('Daily Reward / KPI Monitoring')
plt.xlabel('Day')
plt.ylabel('Reward')
plt.legend()
plt.show()

pd.DataFrame({
    'Metric': ['Mean reward', '95% CI low', '95% CI high', 'CI width'],
    'Value':  [reward_mean, reward_mean - reward_ci, reward_mean + reward_ci, 2*reward_ci]
})

## 6. Failure Diagnosis Dashboard

Automatically flag components of the pipeline that fall outside acceptable thresholds.

In [ ]:
issues = {
    'missing_data_flag':     missing_rate > 0.02,
    'calibration_flag':      brier > 0.20,
    'ab_not_significant':    p_val > 0.05,
    'feature_drift_flag':    ks_p < 0.05,
    'reward_volatile_flag':  reward_ci > 3.0
}

diag_df = pd.DataFrame(list(issues.items()), columns=['Check', 'Flag'])
diag_df['Status'] = diag_df['Flag'].map({True: 'ALERT', False: 'OK'})
diag_df

## 7. Trust Report Summary

A high-level trust report integrates all checks into a readable assessment.

In [ ]:
trust = pd.DataFrame({
    'Dimension': ['Data Quality', 'Calibration', 'A/B Experiment', 'Feature Drift', 'Reward Stability'],
    'Value': [
        f'Missing rate = {missing_rate:.3f}',
        f'Brier = {brier:.3f}',
        f'p = {p_val:.3f}',
        f'KS p = {ks_p:.4f}',
        f'CI width = {2*reward_ci:.2f}'
    ],
    'Status': [
        'Needs attention' if missing_rate > 0.02 else 'OK',
        'Good'           if brier < 0.20           else 'Weak',
        'Validated'      if p_val < 0.05            else 'Uncertain',
        'Drift detected' if ks_p  < 0.05            else 'Stable',
        'Stable'         if reward_ci < 3.0         else 'Volatile'
    ]
})
trust

## 8. Mini Exercises

Try these on your own to complete the capstone:

1. Increase the missing rate to 10% and observe how multiple checks in the dashboard change.
2. Replace Logistic Regression with a Random Forest and compare calibration curves and Brier scores.
3. Change the A/B conversion rates to be closer together (e.g., 0.12 vs 0.13) and compute the minimum sample size required to achieve 80% power.
4. Introduce a larger drift (mean shift of 2.0 instead of 0.5) and verify the KS test detects it more strongly.
5. Add a reward anomaly (a sudden drop to 40 for 5 days) to the KPI monitoring series and update the diagnostic flag logic.
6. Extend the pipeline by adding a resampling-based validation step (k-fold CV) and include cross-validated accuracy in the trust report.
7. Add a Bayesian A/B test using a Beta-Binomial model and compare the posterior probability of B > A with the frequentist p-value.
8. Build a full report as a Pandas DataFrame that includes all seven trust dimensions and their thresholds, then flag any system that fails two or more checks.

This capstone integrates descriptive statistics, hypothesis testing, calibration, drift detection, and reward monitoring — the full statistical toolkit for production AI systems.